# Từ tin tuyển dụng thô đến mô hình dự đoán lương

Notebook đi qua từng bước: biến **tin tuyển dụng** thô thành bảng dữ liệu có **lương** và các thông tin khác (vai trò, cấp bậc, kinh nghiệm, kỹ năng); đo xem máy gán nhãn có chính xác không; rồi **huấn luyện** và **chấm điểm** mô hình (mục 7–10); cuối cùng là cách **duyệt tay** (mục 11).

- Mọi quy tắc gán nhãn nằm trong thư viện `career_market`; notebook chỉ gọi lại để minh họa, không viết lại. Muốn sửa quy tắc thì sửa ở thư viện, notebook tự cập nhật theo.
- Notebook được lưu **không kèm kết quả chạy**: bảng và biểu đồ hiện ra khi bạn chạy.
- Lệnh chính thức để tạo dữ liệu vẫn là `python -m career_market build-dataset`; notebook dùng để trình bày và kiểm tra. Hệ thống kiểm thử tự động chạy thử notebook để nó không bị hỏng.

> **Từ cần biết:** *tập học (train)* là phần dữ liệu mô hình được xem để học; *tập chọn (valid)* dùng để so và chọn mô hình; *tập kiểm tra (test)* chỉ mở một lần ở cuối để chấm điểm thật. *Trung vị* là giá trị đứng giữa khi xếp từ thấp đến cao.

Cách chạy: mở bằng Jupyter hoặc VS Code với môi trường Python của dự án, chọn *Run All*.

In [ ]:
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from career_market.ui import charts  # thư viện biểu đồ chọn backend "Agg" (vẽ cho web)…

%matplotlib inline
# …nên bật lại chế độ inline để biểu đồ hiện ngay trong notebook.

In [ ]:
from career_market.data.label_review import score_review
from career_market.data.dataset import build_dataset, load_dataset, skill_lists
from career_market.data.rules import classify_role, derive_level, extract_skills, parse_experience, parse_salary
from career_market.paths import data_root

pd.set_option("display.max_colwidth", 80)
RAW = data_root() / "raw"
saved = load_dataset()
print("Dữ liệu thô:", sorted(p.name for p in RAW.glob("*.csv")))
print("Bộ dữ liệu đã lưu:", saved.shape)

## 1. Đọc dữ liệu gốc và lọc dần

Hàm `build_dataset` đọc ba nguồn (IT Jobs 2026, TopCV 2026, CareerBuilder 2023) và các lô tin nhập thêm, bỏ tin do máy sinh ra, giữ tin có lương đọc được, giữ tin thuộc **ngành CNTT**, rồi bỏ tin trùng. Bảng "phễu" dưới đây cho biết mỗi bước bỏ đi bao nhiêu tin.

In [ ]:
frame, funnel = build_dataset(RAW)
steps = pd.Series(funnel, name="Số tin").rename(
    {
        "raw": "Tin thô",
        "has_salary": "Có lương đọc được",
        "it_role": "Thuộc vai trò CNTT",
        "deduplicated": "Sau bỏ trùng",
    }
)
display(steps.to_frame())
ax = steps.plot.barh(color="#2563eb", figsize=(6, 2.5))
ax.invert_yaxis()
ax.set_xlabel("Số tin")
plt.show()

# Dựng lại phải ra đúng bộ dữ liệu đã lưu, để nhãn tái lập được.
assert len(frame) == len(saved)
assert frame["salary"].round(3).equals(saved["salary"].round(3))
frame.groupby("source").size().rename("Số tin có nhãn").to_frame()

## 2. Đọc lương

Hàm `parse_salary` đổi chuỗi lương về `(thấp, cao, điểm giữa)` theo đơn vị triệu đồng/tháng: đổi đô la theo tỷ giá năm đăng tin, chia 12 với lương theo năm, bỏ tin ghi "thỏa thuận". Con số mô hình học là **điểm giữa** của khoảng lương.

Hai quy tắc cần biết vì chúng có thể gây sai lệch:
- tin chỉ ghi một đầu ("Tới X", "Từ X"): đầu còn lại được đoán theo thói quen ghi lương của từng trang, học từ những tin ghi đủ hai đầu (xem `docs/research/118-single-salary-rules.md`);
- khoảng có mức cao gấp hơn 3 lần mức thấp bị bỏ, vì không cho biết lương thật.

In [ ]:
examples = [
    "15 - 25 triệu",
    "Lương: 10 Tr - 15 Tr VND",
    "$ 1,200-1,500 /tháng",
    "240tr-360tr ₫/năm",
    "Tới 30 triệu",
    "Từ 20 triệu",
    "Thoả thuận",
    "1 triệu - 40 triệu/tháng",
]
pd.DataFrame(
    [{"Chuỗi lương": text, "Thấp–cao–trung điểm": parse_salary(text)} for text in examples]
)

In [ ]:
ratio = frame["salary_max"] / frame["salary_min"]
one_sided = frame["salary_text"].str.contains(r"tới|đến|up to|từ|trên", case=False, regex=True)
print(f"Tin chỉ ghi một đầu lương: {one_sided.sum()} ({one_sided.mean():.1%})")
print(f"Cao/thấp: trung vị {ratio.median():.2f}, p90 {ratio.quantile(0.9):.2f}")
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
frame["salary"].plot.hist(bins=40, ax=axes[0], color="#2563eb")
axes[0].set(title="Lương trung điểm (triệu/tháng)", ylabel="Số tin")
ratio.plot.hist(bins=30, ax=axes[1], color="#64748b")
axes[1].set(title="Độ rộng khoảng (cao/thấp)", ylabel="Số tin")
plt.show()

## 3. Vai trò, cấp bậc, kinh nghiệm, kỹ năng

- `classify_role`: từ chức danh suy ra một trong 23 vai trò; nếu không phải tin CNTT thì tin bị bỏ.
- `parse_experience`: đổi chuỗi kinh nghiệm thành số năm; "Không yêu cầu" = 0; "1 - 5 năm" lấy giữa là 3.
- `derive_level`: cấp bậc lấy từ chức danh (Senior, Lead…); nếu chức danh không ghi thì suy từ số năm. Không dùng cấp bậc do trang nguồn gắn, vì đo thấy phần lớn nhãn "Intern" của nguồn là sai.
- `extract_skills`: dò danh sách kỹ năng trong chức danh và mô tả công việc.

In [ ]:
titles = [
    "Senior Backend Engineer",
    "MERN Stack Developer",
    "QA Tech Lead",
    "Nhân viên kinh doanh",
    "Product Manager",
    "Data Scientist",
]
pd.DataFrame(
    [
        {"Chức danh": t, "Vai trò": classify_role(t), "Cấp bậc (2 năm)": derive_level(t, 2)}
        for t in titles
    ]
)

In [ ]:
texts = ["Không yêu cầu", "Dưới 1 năm", "Trên 2 Năm", "1 - 5 Năm", "3 năm", "Không hiển thị"]
display(pd.DataFrame([{"Chuỗi kinh nghiệm": t, "Số năm": parse_experience(t)} for t in texts]))
jd = "Yêu cầu: thành thạo Python, SQL; có kinh nghiệm Docker, AWS là lợi thế. API Testing với Postman."
print("Kỹ năng trích từ đoạn JD:", extract_skills(jd))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
frame["role"].value_counts().sort_values().plot.barh(ax=axes[0], color="#2563eb")
axes[0].set(title="Số tin theo vai trò", xlabel="Số tin")
skills = skill_lists(frame).explode().dropna().value_counts().head(20).sort_values()
skills.plot.barh(ax=axes[1], color="#64748b")
axes[1].set(title="20 kỹ năng được nhắc nhiều nhất", xlabel="Số tin")
plt.tight_layout()
plt.show()
frame["level"].value_counts().rename("Số tin").to_frame().T

## 4. Bỏ tin trùng và chia theo công ty

Tin trùng (cùng chức danh, công ty và lương) chỉ giữ một bản. Mỗi công ty nằm trọn trong một tập, nên điểm số đo đúng câu hỏi thật: dự đoán cho **công ty chưa từng gặp**.

In [ ]:
print("Bị loại do trùng:", funnel["it_role"] - funnel["deduplicated"])
display(frame["split"].value_counts().rename("Số tin").to_frame().T)
companies_per_split = frame.groupby("company_key")["split"].nunique()
assert (companies_per_split == 1).all(), "Một công ty không được nằm ở hai tập"
print("Số công ty:", frame["company_key"].nunique(), "· không công ty nào nằm ở hai tập")

## 5. Máy gán nhãn có chính xác không?

Một mẫu 208 tin (chia đều theo nguồn và vai trò) đã được duyệt lại. Lưu ý: mẫu này do **AI duyệt**, chưa thay được người thật duyệt. Muốn duyệt lại: tạo mẫu bằng `career_market.data.label_review.review_sample()`, điền các cột `review_*`, rồi chạy `score_review`.

In [ ]:
review = pd.read_csv(ROOT / "datasets/labels/2026-11-06/label_accuracy_review.csv", dtype=str)
scores = score_review(review)
display(
    pd.DataFrame(
        [
            {
                "Nhãn": k,
                "Số dòng chấm": v["reviewed"],
                "Tỉ lệ đúng (%)": round(v["accuracy_pct"], 1),
            }
            for k, v in scores.items()
            if k != "skills"
        ]
        + [
            {
                "Nhãn": "skills (độ chính xác)",
                "Số dòng chấm": scores["skills"]["assigned"],
                "Tỉ lệ đúng (%)": round(scores["skills"]["precision_pct"], 1),
            }
        ]
    )
)
review.loc[review["review_note"].notna(), ["title", "role", "level", "review_note"]].head(15)

## 6. Xuất bộ dữ liệu

Bản chính thức được ghi bằng `python -m career_market build-dataset`. Ở đây notebook xuất ra một thư mục tạm rồi so với bản đã lưu, để chắc chắn notebook và lệnh xuất cho cùng một kết quả.

In [ ]:
from career_market.data.dataset import write_dataset

output = Path(tempfile.mkdtemp()) / "salary_dataset_v2.csv"
write_dataset(RAW, output)
exported = pd.read_csv(output)
assert exported.shape == saved.shape
assert (exported["row_id"] == saved["row_id"]).all()
print("Đã xuất", exported.shape, "→ khớp bộ dữ liệu đã lưu")

## 7. Huấn luyện

Hàm `train` huấn luyện Ridge, Random Forest và Gradient Boosting trên **tập học**, thử vài thiết lập và giữ thiết lập sai ít nhất trên **tập chọn**. Mô hình cuối cùng cũng được chọn theo tập chọn; **tập kiểm tra** chỉ dùng để báo điểm.

Quy tắc bắt buộc: lương dự đoán không được giảm khi thêm kỹ năng hoặc kinh nghiệm. Notebook huấn luyện vào thư mục tạm, không đụng tới mô hình đang chạy; muốn thay mô hình thật thì dùng `python -m career_market train`, lệnh này tạo phiên bản mới và chỉ dùng nó khi qua cổng kiểm tra.

In [ ]:
from career_market.model.estimators import MODEL_LABELS, feature_frame
from career_market.model.training import train

workdir = Path(tempfile.mkdtemp())
report = train(workdir)
print("Số tin:", report["dataset_rows"], "· chia:", report["split"])
print("Mô hình được chọn (theo valid MAE):", MODEL_LABELS[report["selected"]])
pd.DataFrame(
    [
        {
            "Mô hình": MODEL_LABELS[name],
            "Siêu tham số": str(trial["params"]),
            "MAE valid": round(trial["valid_MAE"], 3),
        }
        for name, trials in report["tuning"].items()
        for trial in trials
    ]
)

In [ ]:
features = feature_frame(frame)
print("Số đặc trưng đầu vào:", features.shape[1])
features.head(3)

## 8. So sánh trên tập học / tập chọn / tập kiểm tra

Nếu điểm trên tập học tốt hơn hẳn hai tập còn lại, mô hình đang "học thuộc" (overfit). "Mốc trung vị" là mức tối thiểu mà mọi mô hình phải vượt.

In [ ]:
rows = []
for name, values in report["models"].items():
    for part in ("train", "valid", "test"):
        rows.append(
            {
                "Mô hình": values["label"],
                "Tập": part,
                **{k: round(values[part][k], 3) for k in ("MAE", "RMSE", "R2", "MAPE")},
            }
        )
scores = pd.DataFrame(rows)
display(scores.pivot(index="Mô hình", columns="Tập", values="MAE")[["train", "valid", "test"]])
ax = scores.pivot(index="Mô hình", columns="Tập", values="MAE")[
    ["train", "valid", "test"]
].plot.barh(figsize=(7, 3.5))
ax.set(xlabel="MAE (triệu VND/tháng)", ylabel="")
plt.show()
scores

## 9. Kết quả trên tập kiểm tra

Mỗi điểm trên biểu đồ là một tin của công ty mô hình chưa từng gặp. Sai số theo vai trò cho biết nhóm nào khiến mô hình đoán kém.

In [ ]:
from career_market.ui import charts

# salary_charts dùng backend "Agg" (vẽ cho web), nên hiển thị bằng display() thay vì plt.show().
predictions = pd.read_csv(workdir / "test_predictions.csv")
for figure in (
    charts.actual_vs_predicted(predictions),
    charts.error_by_role(predictions),
    charts.feature_importance(report),
):
    display(figure)
    plt.close(figure)

In [ ]:
# Huấn luyện có seed cố định: phải ra đúng mô hình đang phục vụ (nếu cùng bộ dữ liệu).
from career_market.model.training import load_report

serving = load_report()
if serving["dataset_rows"] == report["dataset_rows"]:
    selected = report["selected"]
    assert (
        abs(serving["models"][selected]["test"]["MAE"] - report["models"][selected]["test"]["MAE"])
        < 1e-6
    )
    print("Khớp model đang phục vụ: test MAE", round(report["models"][selected]["test"]["MAE"], 3))

## 10. Có đủ tốt để dùng thật?

Chấm theo ba chỉ số dễ hiểu: một nửa số dự đoán lệch dưới bao nhiêu %; bao nhiêu % dự đoán lệch không quá 20%; khoảng 80% có chứa lương thật đúng khoảng 80% số lần không. Để không "đậu nhờ may", mức chung được chấm theo kết quả xấu nhất hợp lý của nhiều lần thi lại theo công ty. Notebook thi lại 200 lần cho nhanh (ứng dụng dùng 1.000 lần), nên con số có thể lệch nhẹ.

In [ ]:
from career_market.model.validation import validate

check = validate(output=workdir / "validation.json", rounds=200, directory=workdir)
production = check["production"]
print("Đạt chuẩn production:", production["production_ready"])
display(
    pd.DataFrame(
        [
            {
                "Chỉ số": name,
                "Giá trị": round(m["point"], 1),
                "Khoảng 95%": "{:.1f} – {:.1f}".format(*m["ci95"]),
                "Ngưỡng": str(m["threshold"]),
                "Đạt": m["pass"],
            }
            for name, m in production["overall"]["metrics"].items()
        ]
    )
)
pd.DataFrame(
    [
        {"Nhóm": row["group"], "Loại": column, "Số tin test": row["n"], "Kết quả": row["status"]}
        for column in ("role", "level")
        for row in production[column]
    ]
)

## 11. Duyệt tay: danh sách chờ và mẫu kiểm tra

Có hai nhóm tin cần người xem lại:

- **Danh sách chờ duyệt**: tin của lô nhập mới mà máy không gán nhãn được (lương "thỏa thuận", lương bất thường, không rõ vai trò). Danh sách được tạo khi chạy `build-dataset`.
- **Mẫu kiểm tra**: khoảng 50 tin máy đã gán nhãn, chọn ngẫu nhiên (cố định để lần sau chọn lại đúng các tin đó), dùng để đo máy gán sai bao nhiêu.

Sửa nhãn bằng `record_review(...)`: quyết định được ghi vào **sổ nhãn đã duyệt** (`datasets/labels/reviewed.csv`) và tự áp dụng ở lần tạo dữ liệu sau.

In [ ]:
from career_market.data.review import load_ledger, load_queue

queue = load_queue()
print(f"Hàng đợi duyệt tay: {len(queue)} tin")
queue.head(20)

In [ ]:
from career_market.data.label_review import review_sample

sample = review_sample(size=50, seed=2026)  # cùng seed → cùng mẫu, để đối chiếu giữa các lần
print(f"Mẫu kiểm tra: {len(sample)} tin (mỗi nguồn tối thiểu 10 tin)")
sample[["row_id", "source", "title", "salary_text", "salary", "role", "level", "skills"]].head(10)

### Máy gán sai bao nhiêu?

Mẫu 67 tin ngày 06/11/2026 do **AI duyệt thay người** (chưa thay được người thật duyệt). Ba lỗi lặp lại nhiều lần đã được sửa thẳng trong quy tắc: JavaScript bị nhận nhầm từ chữ "Node.js"; "Midle" là lỗi chính tả của Middle; Dynamics AX là phần mềm ERP. Hai lỗi chỉ gặp một lần được ghi vào sổ nhãn.

In [ ]:
from career_market.data.review import audit_error_rates
from career_market.paths import data_root

reviewed = pd.read_csv(
    data_root() / "labels/2026-11-06/audit_sample_ai_67.csv", dtype=str, keep_default_na=False
)
display(audit_error_rates(reviewed))
reviewed.loc[reviewed["review_note"] != "", ["title", "role", "level", "review_note"]]

### Sửa một nhãn và thấy kết quả ở lần tạo dữ liệu sau

Ví dụ dưới ghi vào một **sổ tạm**, để chạy lại notebook không làm thay đổi sổ thật. Khi duyệt thật, bỏ tham số `path` để ghi vào `datasets/labels/reviewed.csv`.

In [ ]:
import tempfile

from career_market.data.dataset import build_dataset, load_dataset
from career_market.data.review import record_review, review_key

row = load_dataset().query("title == 'Nhân viên IT'").iloc[0]
key = review_key(row["source_url"], row["title"], row["company"])
ledger = Path(tempfile.mkdtemp()) / "reviewed.csv"
record_review(key, "accept", path=ledger, role="IT Support / Helpdesk", level="Fresher", reviewer="ví dụ")
rebuilt, _ = build_dataset(ledger=ledger)
after = rebuilt[rebuilt["source_url"] == row["source_url"]].iloc[0]
print("Trước:", row["role"], "·", row["level"], "→ Sau:", after["role"], "·", after["level"])
print("Sổ nhãn thật hiện có", len(load_ledger()), "quyết định")